In [1]:
# ============================================================
# F&B 메뉴별 매출수량 예측 — All-in-One (LB 정합 최적화 + LSTM 통합)
#  - Train → Multi-bank Validate → Constrained Meta-blend → Test → Submission
#  - 핵심: (1) 미래 달력 피처 일치 (2) 리더보드형 sMAPE (A=0 제외, 업장 가중)
#  - 안전장치: NumPy 2.x 호환, 인코딩 감지, 경로 자동화, all-zero 가드
#  - 모델: Global XGB Ens, Local XGB(by store), CatBoost(Poisson), LGB Horizon(+Quantile),
#          sNaive, Pos-only XGB, **Global LSTM (신규)**
# ============================================================
import sys, subprocess, os, re, gc, math, random, glob, unicodedata, warnings
warnings.filterwarnings("ignore")

# --- (Colab 환경만) 드라이브 마운트 시도 ---
try:
    from google.colab import drive  # type: ignore
    try:
        drive.mount('/content/drive')
    except Exception:
        pass
except Exception:
    pass

def _pip_install(pkgs):
    try:
        import importlib
        for p in pkgs:
            _ = importlib.import_module(p if p!="xgboost" else "xgboost")
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)

# 최소 필요 라이브러리
_pip_install(["xgboost","lightgbm","catboost","scikit-learn","scipy","tqdm","pandas","numpy"])

import numpy as np, pandas as pd
import xgboost as xgb
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
import lightgbm as lgb
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Versions | xgboost:", xgb.__version__, "| lightgbm:", lgb.__version__, "| GPU:", torch.cuda.is_available())

# ------------------------------------------------------------
# 경로/기본 하이퍼
# ------------------------------------------------------------
def _auto_path(cands):
    for p in cands:
        if isinstance(p, str) and os.path.exists(p):
            return p
    return cands[0]

# 가능한 경로들(Colab/로컬/플랫폼 폴백)
TRAIN_PATH = _auto_path([
    "/content/drive/MyDrive/data/lg/train/train.csv",
    "/content/drive/MyDrive/data/lg/train.csv",
    "/content/drive/MyDrive/train.csv",
    "/mnt/data/train.csv",
    "train.csv",
])
TEST_DIR = _auto_path([
    "/content/drive/MyDrive/data/lg/test",
    "/mnt/data",
    "test",
    ".",
])
SAMPLE_SUB = _auto_path([
    "/content/drive/MyDrive/data/lg/sample_submission.csv",
    "/mnt/data/sample_submission.csv",
    "sample_submission.csv",
])
SAVE_PATH  = _auto_path([
    "/content/drive/MyDrive/data/lg/submission_robust_opt_lstm.csv",
    "submission_robust_opt_lstm.csv"
])

# 윈도우/검증
LOOKBACK, PREDICT = 28, 7
VALID_DAYS        = 56
SEED              = 42

# ===== [권장 상수] =====
BLEND_STEP        = 0.10
CD_REFINE_STEP    = 0.02
CD_REFINE_ROUNDS  = 10
SNAIVE_MAX_W      = 0.12
SNAIVE_CAP_MAX    = 0.35
ALL_ZERO_GUARD    = True
XGB_ENS_SEEDS     = [41, 42, 777, 1337, 2024, 31415]

# 리더보드 업장 가중
LB_STORE_W = {"담하": 1.9, "미라시아": 1.6}

# 학습 샘플 가중
STORE_PRIORITY = {"담하": 1.9, "미라시아": 1.6}

# ε-floor
USE_DYNAMIC_EPSILON = True
EPS_MIN_FLOOR      = 0.20

# Soft Cap 설정
USE_SOFT_CAP = True
CAP_LOOKBACK = 42
CAP_PERC     = 90.0
CAP_MUL      = 1.12
CAP_MIN      = 0.0
CAP_MAX      = None

# 게이팅(부드러운 보정)
GATE_A, GATE_B = 0.90, 0.10
SQUASH_A       = 0.035

# 전역 제약 메타블렌더
USE_PER_MENU_WEIGHTS   = True
USE_GLOBAL_CONSTRAINED = True
GLOBAL_SNAIVE_CAP      = 0.30

# CatBoost Poisson
USE_CATBOOST_POISSON = True

# ===== LSTM 사용/하이퍼 =====
USE_LSTM_GLOBAL           = True     # ← LSTM 사용 스위치
LSTM_POSREG_BLEND_ALPHA   = 0.45     # ← Yr(Pos-only XGB)와 LSTM 블렌드 비율 (0~1)
LSTM_HIDDEN_SIZE          = 128
LSTM_NUM_LAYERS           = 2
LSTM_DROPOUT              = 0.10     # num_layers>1일 때만 적용
LSTM_LR                   = 1e-3
LSTM_BATCH_SIZE           = 1024
LSTM_EPOCHS               = 25
LSTM_PATIENCE             = 5
LSTM_WEIGHT_CLIP          = 1.0      # grad clip

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); os.environ['PYTHONHASHSEED']=str(seed)
    try:
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
            torch.backends.cudnn.deterministic = True
            torch.backends.cudnn.benchmark = False
    except Exception:
        pass
set_seed(SEED)

# ------------------------------------------------------------
# IO utils & cleaning
# ------------------------------------------------------------
def read_csv_safe(p):
    for enc in ["utf-8-sig","cp949","euc-kr","utf-8"]:
        try:
            return pd.read_csv(p, encoding=enc)
        except Exception:
            pass
    return pd.read_csv(p)

def normalize_name(s: str) -> str:
    if not isinstance(s, str): s = str(s)
    s = unicodedata.normalize('NFKC', s)
    s = s.replace('\xa0',' ')
    s = re.sub(r'\s+',' ', s).strip()
    return s

def clean_df(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['영업일자'] = pd.to_datetime(df['영업일자'])
    df['영업장명_메뉴명'] = df['영업장명_메뉴명'].astype(str).map(normalize_name)
    df['매출수량'] = pd.to_numeric(df['매출수량'], errors='coerce').fillna(0).clip(lower=0)
    return df

train_raw = clean_df(read_csv_safe(TRAIN_PATH)).sort_values(['영업장명_메뉴명','영업일자']).reset_index(drop=True)
parted = train_raw['영업장명_메뉴명'].astype(str).str.partition('_')
train_raw['업장'] = parted[0].str.strip()

date_min, date_max = train_raw['영업일자'].min(), train_raw['영업일자'].max()
cut_date = date_max - pd.Timedelta(days=VALID_DAYS)
print(f"Train: {date_min.date()} ~ {date_max.date()} | Valid from: {cut_date.date()}")

# ------------------------------------------------------------
# Feature Engineering
# ------------------------------------------------------------
FIXED_SOLAR_HOLS = {(1,1),(3,1),(5,5),(8,15),(10,3),(12,25)}

CALENDAR_COLS = [
    'dow','month','is_weekend','is_holiday','is_big_holiday_window',
    'dow_sin','dow_cos','woy','woy_sin','woy_cos',
] + [f'dow_{d}' for d in range(7)]

def _to_dt_series(dts) -> pd.Series:
    if isinstance(dts, pd.Series):
        return pd.to_datetime(dts)
    return pd.Series(pd.to_datetime(dts))

def _make_calendar_from_dates(dts) -> pd.DataFrame:
    dts = _to_dt_series(dts)
    cal = pd.DataFrame(index=dts.index)
    cal['dow']   = dts.dt.weekday
    cal['month'] = dts.dt.month
    cal['is_weekend'] = (cal['dow']>=5).astype(int)
    cal['is_holiday'] = [int((int(m), int(d)) in FIXED_SOLAR_HOLS) for m,d in zip(cal['month'], dts.dt.day)]
    cal['is_big_holiday_window'] = (
        ((cal['month']==1) & (dts.dt.day>=20)) |
        ((cal['month']==2) & (dts.dt.day<=15)) |
        ((cal['month']==9) & (dts.dt.day>=10)) |
        ((cal['month']==10)& (dts.dt.day<=10))
    ).astype(int)
    cal['dow_sin'] = np.sin(2*np.pi * cal['dow']/7.0)
    cal['dow_cos'] = np.cos(2*np.pi * cal['dow']/7.0)
    iso = dts.dt.isocalendar()
    try:    woy = iso.week.astype(int)
    except: woy = iso['week'].astype(int)
    cal['woy'] = woy
    cal['woy_sin'] = np.sin(2*np.pi * woy/52.0)
    cal['woy_cos'] = np.cos(2*np.pi * woy/52.0)
    for d in range(7):
        cal[f'dow_{d}'] = (cal['dow']==d).astype(int)
    return cal

def add_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values(['영업장명_메뉴명','영업일자']).copy()
    cal = _make_calendar_from_dates(df['영업일자'])
    df = pd.concat([df.reset_index(drop=True), cal.reset_index(drop=True)], axis=1)

    grp = df.groupby('영업장명_메뉴명')['매출수량']
    for lag in (1,7,14):
        df[f'lag_{lag}'] = grp.shift(lag)
    for win in (7,14):
        df[f'roll_mean_{win}'] = grp.shift(1).rolling(win, min_periods=1).mean()

    df['diff_1']    = grp.diff(1).clip(-50,50)
    df['pct_chg_1'] = grp.pct_change(1).clip(-2,2)

    def _by_group(g):
        y = g['매출수량']
        g['sum_7']     = y.shift(1).rolling(7, 1).sum()
        g['sum_14']    = y.shift(1).rolling(14,1).sum()
        g['mean_14']   = y.shift(1).rolling(14,1).mean()
        g['sum_prev7'] = y.shift(8).rolling(7, 1).sum()

        z = (y==0).astype(int)
        g['zero_share_7']  = z.shift(1).rolling(7, 1).mean()
        g['zero_share_28'] = z.shift(1).rolling(28,1).mean()

        idx = np.arange(len(g))
        nz_prev = (y>0).shift(1).fillna(0).astype(int).to_numpy()
        last = np.where(nz_prev==1, idx, np.nan)
        last = pd.Series(last, index=g.index).ffill().fillna(-1).to_numpy()
        g['nonzero_gap'] = (idx - last).astype('float32')

        g['zero_rate7']  = (y.eq(0).shift(1).rolling(7, min_periods=3).mean()).fillna(0.0)
        g['wknd_x_mean7']  = g['is_weekend'] * g['mean_14'].fillna(0.0)
        g['holi_x_mean14'] = g['is_holiday'] * g['mean_14'].fillna(0.0)
        g['roll_med_7']    = y.shift(1).rolling(7, 1).median()

        g.replace([np.inf,-np.inf], np.nan, inplace=True)
        g.fillna(0, inplace=True)
        return g

    df = df.groupby('영업장명_메뉴명', group_keys=False).apply(_by_group)
    return df

train_feat = add_features(train_raw)

FEATURE_COLS = [
    '매출수량',
    # calendar
    'dow','month','is_weekend','is_holiday','is_big_holiday_window',
    'dow_sin','dow_cos','woy','woy_sin','woy_cos',
    # one-hot
    'dow_0','dow_1','dow_2','dow_3','dow_4','dow_5','dow_6',
    # time-series
    'lag_1','lag_7','lag_14','roll_mean_7','roll_mean_14','diff_1','pct_chg_1',
    'sum_7','sum_14','mean_14','sum_prev7','zero_share_7','zero_share_28',
    'nonzero_gap','zero_rate7','wknd_x_mean7','holi_x_mean14','roll_med_7',
]
print("n_features:", len(FEATURE_COLS))

# ------------------------------------------------------------
# Windows & Split & Metric (LB 정합)
# ------------------------------------------------------------
def build_windows(df_feat, lookback=28, predict=7):
    X, Y, stores, menus, tend = [], [], [], [], []
    for menu, g in df_feat.groupby('영업장명_메뉴명'):
        g = g.sort_values('영업일자')
        arr = g[FEATURE_COLS].to_numpy(dtype=np.float32)
        if len(arr) < lookback+predict: continue
        store = menu.split("_",1)[0].strip()
        dates = g['영업일자'].values
        for i in range(len(arr)-lookback-predict+1):
            X.append(arr[i:i+lookback].reshape(-1))
            Y.append(arr[i+lookback:i+lookback+predict, 0])
            stores.append(store); menus.append(menu); tend.append(dates[i+lookback+predict-1])
    X = np.stack(X).astype(np.float32)
    Y = np.stack(Y).astype(np.float32)
    meta = pd.DataFrame({'업장':stores, '영업장명_메뉴명':menus, 'target_end':tend})
    return X, Y, meta

X_all, Y_all, meta_all = build_windows(train_feat, LOOKBACK, PREDICT)
is_tr = (meta_all['target_end'] <= cut_date).values
X_tr, Y_tr, meta_tr = X_all[is_tr], Y_all[is_tr], meta_all[is_tr]
X_va, Y_va, meta_va = X_all[~is_tr], Y_all[~is_tr], meta_all[~is_tr]
print("Train/Valid windows:", len(X_tr), len(X_va))

# 업장 가중 + A=0 제외 sMAPE (리더보드 정합)
def smape_ignore_zeros_per_item(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true, float); y_pred = np.asarray(y_pred, float)
    mask = (y_true > 0)
    Ti = int(mask.sum())
    if Ti == 0: return np.nan, 0
    s = 2.0*np.abs(y_true[mask]-y_pred[mask])/(np.abs(y_true[mask])+np.abs(y_pred[mask])+eps)
    return float(np.mean(s)), Ti

def leaderboard_smape(df_long, store_weights=None):
    sw = store_weights or {}
    num, den = 0.0, 0.0
    for store, g_s in df_long.groupby('업장'):
        item_scores=[]
        for _, g_i in g_s.groupby('영업장명_메뉴명'):
            s_i, Ti = smape_ignore_zeros_per_item(g_i['A'].values, g_i['P'].values)
            if Ti > 0 and not np.isnan(s_i): item_scores.append(s_i)
        if not item_scores: continue
        store_avg = float(np.mean(item_scores))
        w = float(sw.get(store, 1.0))
        num += w * store_avg; den += w
    return float(num/den) if den>0 else float('nan')

# baseline sMAPE (window-level, 참고용)
def smape_eval(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true, float); y_pred = np.asarray(y_pred, float)
    mask = (y_true > 0)
    if mask.sum()==0: return 0.0
    num = 2.0*np.abs(y_true[mask]-y_pred[mask])
    den = np.abs(y_true[mask]) + np.abs(y_pred[mask]) + eps
    return float(np.mean(num/den))

# sNaive: 최근 4주 동일요일 중앙값
def snaive7_from_window(x_flat):
    F = len(FEATURE_COLS); x = x_flat.reshape(LOOKBACK, F)
    y = x[:,0].copy()
    week_cols = [y[i::7] for i in range(7)]
    med = np.array([np.median(w[-4:]) if len(w)>=4 else (w[-1] if len(w)>0 else 0.0) for w in week_cols], dtype=float)
    return np.maximum(med, 0.0)

# 학습 샘플 가중: 업장 중요도 × 양일 비중 × 최근성
w_store_tr = meta_tr['업장'].map(lambda s: STORE_PRIORITY.get(s, 1.0)).astype(np.float32).values
pos_ratio_tr = (Y_tr > 0).mean(axis=1).astype(np.float32)
w_pos_tr     = 0.75 + 0.25*pos_ratio_tr
tmax = meta_tr['target_end'].max()
rec_days = (tmax - meta_tr['target_end']).dt.days.to_numpy(dtype=np.float32)
rec_w = 1.0 + 0.40*(1.0 - (rec_days - rec_days.min())/(np.ptp(rec_days)+1e-6))
W_TR = (w_store_tr * w_pos_tr * rec_w).astype(np.float32)
print("Sample weight | min/mean/max:", float(W_TR.min()), float(W_TR.mean()), float(W_TR.max()))

# ------------------------------------------------------------
# Models (XGB / Cat / LGB / PosReg)
# ------------------------------------------------------------
def make_xgb(params=None, seed=42):
    major = int(xgb.__version__.split('.')[0])
    p = dict(
        n_estimators=1600, learning_rate=0.03,
        max_depth=6, subsample=0.90, colsample_bytree=0.90,
        min_child_weight=3, gamma=0.15,
        reg_alpha=0.4, reg_lambda=2.0,
        random_state=seed,
        objective='reg:tweedie', tweedie_variance_power=1.30,
        eval_metric='rmse', n_jobs=-1, tree_method='hist',
    )
    if torch.cuda.is_available() and major >= 2:
        p['device'] = 'cuda'
    if params: p.update(params)
    return XGBRegressor(**p)

class XGBMulti:
    def __init__(self, base_params=None, seed=42):
        self.base_params = base_params or {}
        self.seed = seed
        self.models=[]
    def fit(self, X_tr, Y_tr, X_va=None, Y_va=None, sample_weight=None):
        sw = sample_weight.astype(np.float32) if sample_weight is not None else None
        self.models=[]
        for i in range(Y_tr.shape[1]):
            m = make_xgb(self.base_params, seed=self.seed+i)
            if X_va is not None and Y_va is not None:
                m.fit(X_tr, Y_tr[:,i], eval_set=[(X_va, Y_va[:,i])], sample_weight=sw, verbose=False)
            else:
                m.fit(X_tr, Y_tr[:,i], sample_weight=sw, verbose=False)
            self.models.append(m)
        return self
    def predict(self, X):
        preds = [m.predict(X) for m in self.models]
        return np.column_stack(preds).clip(0, None)

class XGBEnsembleMulti:
    def __init__(self, seeds, base_params=None):
        self.seeds=list(seeds); self.base_params=base_params or {}; self.members=[]
    def fit(self, X_tr, Y_tr, X_va=None, Y_va=None, sample_weight=None):
        self.members=[]
        for sd in self.seeds:
            m = XGBMulti(base_params=self.base_params, seed=sd)
            m.fit(X_tr, Y_tr, X_va, Y_va, sample_weight)
            self.members.append(m)
        return self
    def predict(self, X):
        return np.mean([m.predict(X) for m in self.members], axis=0)

# Positive-only 회귀 (log-transform)
class XGBPosMulti:
    def __init__(self, base_params=None, seed=777):
        self.base_params = base_params or {}
        self.seed=seed
        self.models=[]
    def fit(self, X_tr, Y_tr, X_va=None, Y_va=None):
        self.models=[]
        for i in range(Y_tr.shape[1]):
            mask_tr = (Y_tr[:,i] > 0)
            m = make_xgb(self.base_params, seed=self.seed+100+i)
            if mask_tr.sum() < 50:
                m.fit(X_tr, np.log1p(np.maximum(Y_tr[:,i],0.0)))
                self.models.append(m); continue
            if X_va is not None and Y_va is not None:
                mask_va = (Y_va[:,i] > 0)
                if mask_va.sum() >= 20:
                    m.fit(X_tr[mask_tr], np.log1p(Y_tr[mask_tr,i]),
                          eval_set=[(X_va[mask_va], np.log1p(Y_va[mask_va,i]))], verbose=False)
                else:
                    m.fit(X_tr[mask_tr], np.log1p(Y_tr[mask_tr,i]))
            else:
                m.fit(X_tr[mask_tr], np.log1p(Y_tr[mask_tr,i]))
            self.models.append(m)
        return self
    def predict(self, X):
        preds = [np.expm1(m.predict(X)) for m in self.models]
        return np.column_stack(preds).clip(0, None)

# ---- LightGBM Horizon / Quantile (미래 달력 주입) ----
class _LGBBase:
    def __init__(self):
        self.model=None
        self.features=None
        self.cat_feats=['store','menu']
        self.store_levels=None
        self.menu_levels=None
    def _prepare_train(self, df):
        df = df.sort_values(['영업장명_메뉴명','영업일자']).copy()
        s = df['영업장명_메뉴명'].astype(str)
        sp = s.str.split(pat='_', n=1, expand=True)
        df['store'] = sp[0]
        df['menu']  = sp[1].fillna('UNK')
        self.store_levels = sorted(df['store'].unique().tolist())
        self.menu_levels  = sorted(df['menu'].unique().tolist())
        df['store'] = pd.Categorical(df['store'], categories=self.store_levels)
        df['menu']  = pd.Categorical(df['menu'],  categories=self.menu_levels)
        return df
    def _inject_future_calendar(self, Xh, base_dates, h):
        fut_dates = base_dates + pd.to_timedelta(h, unit="D")
        fut_cal = _make_calendar_from_dates(fut_dates).reset_index(drop=True)
        Xh = Xh.reset_index(drop=True)
        for c in CALENDAR_COLS:
            if c in Xh.columns:
                Xh[c] = fut_cal[c].astype(Xh[c].dtype if c in Xh.columns else np.float32)
        return Xh
    def _build_supervised_rows(self, df, num_feats):
        X_rows, y_rows = [], []
        for _, g in df.groupby('영업장명_메뉴명'):
            g = g.sort_values('영업일자')
            if len(g) < LOOKBACK + PREDICT: continue
            for h in range(1, PREDICT+1):
                target = g['매출수량'].shift(-h); valid = g.index[:-h]
                Xh = g.loc[valid, num_feats + self.cat_feats].copy()
                Xh['horizon']=h
                Xh = self._inject_future_calendar(Xh, g.loc[valid, '영업일자'], h)
                X_rows.append(Xh); y_rows.append(target.loc[valid].values)
        X = pd.concat(X_rows, ignore_index=True) if X_rows else pd.DataFrame(columns=num_feats+self.cat_feats+['horizon'])
        y = np.concatenate(y_rows) if y_rows else np.zeros(0, dtype=float)
        for c in self.cat_feats:
            X[c] = pd.Categorical(X[c], categories=(self.store_levels if c=='store' else self.menu_levels))
        return X, y
    def _make_Xh_last(self, g_lastrow):
        base = g_lastrow.iloc[-1]
        rows=[]
        for h in range(1, PREDICT+1):
            r = {k: float(base[k]) for k in FEATURE_COLS}
            r['store']= str(base['store'])
            r['menu'] = str(base['menu'])
            r['horizon']=h
            rows.append(r)
        Xh = pd.DataFrame(rows)
        last_date = g_lastrow.iloc[-1]['영업일자']
        fut_dates = pd.to_datetime([last_date + pd.Timedelta(days=h) for h in range(1, PREDICT+1)])
        fut_cal = _make_calendar_from_dates(fut_dates).reset_index(drop=True)
        for c in CALENDAR_COLS:
            if c in Xh.columns:
                Xh[c] = fut_cal[c].astype(Xh[c].dtype if c in Xh.columns else np.float32)
        Xh['store'] = pd.Categorical(Xh['store'], categories=self.store_levels)
        Xh['menu']  = pd.Categorical(Xh['menu'],  categories=self.menu_levels)
        return Xh

class LGBHorizon(_LGBBase):
    def __init__(self, params=None, n_estimators=2200, random_state=42):
        super().__init__()
        self.params = params or dict(objective='rmse', metric='rmse',
            learning_rate=0.045, num_leaves=255, min_data_in_leaf=30,
            feature_fraction=0.9, bagging_fraction=0.8, bagging_freq=1,
            lambda_l2=1.2, max_depth=-1, random_state=random_state, verbosity=-1)
        self.n_estimators=n_estimators
        self.model = lgb.LGBMRegressor(**self.params, n_estimators=self.n_estimators)
    def fit(self, train_feat_df: pd.DataFrame):
        df = self._prepare_train(train_feat_df)
        num_feats = FEATURE_COLS
        X, y = self._build_supervised_rows(df, num_feats)
        self.features = num_feats + self.cat_feats + ['horizon']
        if len(X)==0: self.model=None; return self
        self.model.fit(X[self.features], y, categorical_feature=self.cat_feats)
        return self
    def predict_lastrow(self, g_lastrow: pd.DataFrame):
        if self.model is None: return np.zeros(PREDICT, dtype=float)
        Xh = self._make_Xh_last(g_lastrow)
        preds = self.model.predict(Xh[self.features])
        return np.maximum(preds, 0.0)

class LGBQuantileHorizon(_LGBBase):
    def __init__(self, alpha=0.10, n_estimators=1500, random_state=202):
        super().__init__()
        self.params = dict(objective='quantile', alpha=alpha,
            learning_rate=0.045, num_leaves=127, min_data_in_leaf=25,
            feature_fraction=0.9, bagging_fraction=0.8, bagging_freq=1,
            lambda_l2=1.0, max_depth=-1, random_state=random_state, verbosity=-1)
        self.n_estimators=n_estimators
        self.model = lgb.LGBMRegressor(**self.params, n_estimators=self.n_estimators)
    def fit(self, train_feat_df: pd.DataFrame):
        df = self._prepare_train(train_feat_df)
        num_feats = FEATURE_COLS
        X, y = self._build_supervised_rows(df, num_feats)
        self.features = num_feats + self.cat_feats + ['horizon']
        if len(X)==0: self.model=None; return self
        self.model.fit(X[self.features], y, categorical_feature=self.cat_feats)
        return self
    def predict_lastrow(self, g_lastrow: pd.DataFrame):
        if self.model is None: return np.zeros(PREDICT, dtype=float)
        Xh = self._make_Xh_last(g_lastrow)
        preds = self.model.predict(Xh[self.features])
        return np.maximum(preds, 0.0)

# ------------------------------------------------------------
# (신규) Global LSTM 모델
# ------------------------------------------------------------
class LSTMSeqReg(nn.Module):
    def __init__(self, in_dim, hidden=128, layers=2, dropout=0.1, out_dim=PREDICT):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=in_dim,
            hidden_size=hidden,
            num_layers=layers,
            batch_first=True,
            dropout=(dropout if layers>1 else 0.0)
        )
        self.head = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, out_dim)
        )
    def forward(self, x):  # x: (B, T, F)
        h, _ = self.lstm(x)           # (B, T, H)
        last = h[:, -1, :]            # (B, H)
        out = self.head(last)         # (B, PREDICT)
        return out

class GlobalLSTM:
    def __init__(self,
                 in_dim,
                 hidden=LSTM_HIDDEN_SIZE,
                 layers=LSTM_NUM_LAYERS,
                 dropout=LSTM_DROPOUT,
                 lr=LSTM_LR,
                 batch_size=LSTM_BATCH_SIZE,
                 epochs=LSTM_EPOCHS,
                 patience=LSTM_PATIENCE,
                 grad_clip=LSTM_WEIGHT_CLIP,
                 device=DEVICE):
        self.in_dim=in_dim; self.hidden=hidden; self.layers=layers; self.dropout=dropout
        self.lr=lr; self.batch_size=batch_size; self.epochs=epochs; self.patience=patience; self.grad_clip=grad_clip
        self.device=device
        self.model=None
        self.mu=None; self.std=None

    @staticmethod
    def _to_seq(X_flat):
        F = len(FEATURE_COLS)
        return X_flat.reshape(-1, LOOKBACK, F)

    def _fit_scaler(self, X_seq):
        X2 = X_seq.reshape(-1, X_seq.shape[-1])
        self.mu = X2.mean(axis=0).astype(np.float32)
        self.std = X2.std(axis=0).astype(np.float32)
        self.std = np.where(self.std<1e-6, 1e-6, self.std)

    def _scale(self, X_seq):
        return (X_seq - self.mu) / self.std

    def fit(self, X_tr_flat, Y_tr, X_va_flat=None, Y_va=None, sample_weight=None):
        Xtr = self._to_seq(X_tr_flat).astype(np.float32)
        self._fit_scaler(Xtr)
        Xtr = self._scale(Xtr)

        ytr = Y_tr.astype(np.float32)

        if X_va_flat is not None and Y_va is not None:
            Xva = self._scale(self._to_seq(X_va_flat).astype(np.float32))
            yva = Y_va.astype(np.float32)
        else:
            Xva, yva = None, None

        # 가중치 정규화
        if sample_weight is None:
            wtr = np.ones(len(Xtr), dtype=np.float32)
        else:
            wtr = sample_weight.astype(np.float32)
            wtr = wtr / (wtr.mean() + 1e-6)

        tr_ds = TensorDataset(
            torch.from_numpy(Xtr), torch.from_numpy(ytr), torch.from_numpy(wtr)
        )
        tr_loader = DataLoader(tr_ds, batch_size=self.batch_size, shuffle=True, drop_last=False)

        if Xva is not None:
            va_ds = TensorDataset(torch.from_numpy(Xva), torch.from_numpy(yva))
            va_loader = DataLoader(va_ds, batch_size=self.batch_size, shuffle=False, drop_last=False)
        else:
            va_loader=None

        self.model = LSTMSeqReg(self.in_dim, self.hidden, self.layers, self.dropout, out_dim=PREDICT).to(self.device)
        opt = torch.optim.Adam(self.model.parameters(), lr=self.lr)

        def weighted_mse(pred, target, w):
            # pred/target: (B, P), w: (B,)
            mse_sample = ((pred - target)**2).mean(dim=1)
            return (mse_sample * w).mean()

        best_loss = float('inf'); wait=0
        best_state = None

        for epoch in range(self.epochs):
            self.model.train()
            tr_loss=0.0; nbt=0
            for xb, yb, wb in tr_loader:
                xb = xb.to(self.device); yb = yb.to(self.device); wb=wb.to(self.device)
                opt.zero_grad(set_to_none=True)
                out = self.model(xb)
                loss = weighted_mse(out, yb, wb)
                loss.backward()
                if self.grad_clip is not None:
                    nn.utils.clip_grad_norm_(self.model.parameters(), self.grad_clip)
                opt.step()
                tr_loss += float(loss.item()); nbt += 1
            tr_loss = tr_loss/max(nbt,1)

            if va_loader is not None:
                self.model.eval()
                va_loss=0.0; nbv=0
                with torch.no_grad():
                    for xb, yb in va_loader:
                        xb = xb.to(self.device); yb = yb.to(self.device)
                        out = self.model(xb)
                        loss = ((out - yb)**2).mean()
                        va_loss += float(loss.item()); nbv += 1
                va_loss = va_loss/max(nbv,1)
                if va_loss < best_loss - 1e-6:
                    best_loss = va_loss; wait=0
                    best_state = {k:v.detach().cpu().clone() for k,v in self.model.state_dict().items()}
                else:
                    wait += 1
                if wait >= self.patience:
                    break
            else:
                # 검증 없으면 마지막 것을 사용
                best_state = {k:v.detach().cpu().clone() for k,v in self.model.state_dict().items()}

        if best_state is not None:
            self.model.load_state_dict(best_state)

        return self

    def predict(self, X_flat):
        assert self.model is not None, "LSTM model not fitted"
        X = self._scale(self._to_seq(X_flat).astype(np.float32))
        self.model.eval()
        preds=[]
        with torch.no_grad():
            for i in range(0, len(X), self.batch_size):
                xb = torch.from_numpy(X[i:i+self.batch_size]).to(self.device)
                out = self.model(xb).cpu().numpy()
                preds.append(out)
        P = np.vstack(preds).astype(np.float32)
        return np.clip(P, 0.0, None)  # 음수 방지
# ------------------------------------------------------------

# ------------------------------------------------------------
# Train base channels
# ------------------------------------------------------------
gc.collect()

# 1) Global XGB 앙상블
global_xgb_ens = XGBEnsembleMulti(seeds=XGB_ENS_SEEDS)
global_xgb_ens.fit(X_tr, Y_tr, X_va, Y_va, sample_weight=W_TR)
Y_va_g = np.clip(global_xgb_ens.predict(X_va), 0, None)
print("Valid sMAPE | Global XGB ens [window-level]:", round(smape_eval(Y_va, Y_va_g), 4)); gc.collect()

# 2) Local XGB (업장별)
MIN_LOCAL_WINS = 50
local_models = {}
for store in tqdm(sorted(meta_tr['업장'].unique()), desc="Local XGB (by store)"):
    m_tr = (meta_tr['업장'].values == store)
    if m_tr.sum() < MIN_LOCAL_WINS: continue
    mdl = XGBMulti(base_params=dict(n_estimators=1200, max_depth=6, learning_rate=0.035), seed=SEED)
    m_va = (meta_va['업장'].values == store)
    if m_va.sum() > 0:
        mdl.fit(X_tr[m_tr], Y_tr[m_tr], X_va[m_va], Y_va[m_va], sample_weight=W_TR[m_tr])
    else:
        mdl.fit(X_tr[m_tr], Y_tr[m_tr], sample_weight=W_TR[m_tr])
    local_models[store] = mdl
print("Local XGB models:", len(local_models))

# 3) Global CatBoost (Poisson)
def make_cat_model(random_state=SEED):
    return CatBoostRegressor(
        iterations=2200, learning_rate=0.035, depth=7,
        loss_function='Poisson', eval_metric='Poisson',
        l2_leaf_reg=8.0, random_strength=1.0,
        leaf_estimation_iterations=6,
        random_state=random_state, verbose=False,
        task_type=('GPU' if torch.cuda.is_available() else 'CPU'),
        od_type='Iter', od_wait=250, allow_writing_files=False
    )

Y_va_c = np.zeros_like(Y_va)
cat_models=[]
for h in range(PREDICT):
    m = make_cat_model(SEED + h)
    m.fit(X_tr, Y_tr[:,h], eval_set=(X_va, Y_va[:,h]), sample_weight=W_TR, use_best_model=True, verbose=False)
    Y_va_c[:,h] = np.clip(m.predict(X_va), 0, None)
    cat_models.append(m)
print("Valid sMAPE | Global CAT (Poisson) [window-level]:", round(smape_eval(Y_va, Y_va_c), 4))

# 4) LGB Horizon + Quantiles (P10, P50, P90)
lgb_h   = LGBHorizon().fit(train_feat)
lgb_q10 = LGBQuantileHorizon(alpha=0.10, n_estimators=1500).fit(train_feat)
lgb_q50 = LGBQuantileHorizon(alpha=0.50, n_estimators=1500).fit(train_feat)
lgb_q90 = LGBQuantileHorizon(alpha=0.90, n_estimators=1500).fit(train_feat)

# Horizon 검증 예측
df_sorted = train_feat.sort_values(['영업장명_메뉴명','영업일자']).copy()
s = df_sorted['영업장명_메뉴명'].astype(str)
sp = s.str.split('_', n=1, expand=True)
df_sorted['store'] = sp[0]; df_sorted['menu'] = sp[1].fillna('UNK')

Y_va_h   = np.zeros_like(Y_va)
Y_va_q10 = np.zeros_like(Y_va)
Y_va_q50 = np.zeros_like(Y_va)
Y_va_q90 = np.zeros_like(Y_va)
for i in range(len(meta_va)):
    menu = meta_va.iloc[i]['영업장명_메뉴명']
    tgt_end = pd.to_datetime(meta_va.iloc[i]['target_end'])
    ref_date = tgt_end - pd.Timedelta(days=PREDICT-1)
    g = df_sorted[df_sorted['영업장명_메뉴명']==menu]
    g2 = g[g['영업일자']<=ref_date]
    if len(g2)>0:
        Y_va_h[i]   = lgb_h.predict_lastrow(g2)
        Y_va_q10[i] = lgb_q10.predict_lastrow(g2)
        Y_va_q50[i] = lgb_q50.predict_lastrow(g2)
        Y_va_q90[i] = lgb_q90.predict_lastrow(g2)
print("Valid sMAPE | LGB horizon [window-level]:", round(smape_eval(Y_va, Y_va_h), 4))

# 5) sNaive
Y_va_n = np.stack([snaive7_from_window(x) for x in X_va])

# 6) Positive-only 회귀
pos_reg = XGBPosMulti(base_params=dict(n_estimators=900, max_depth=6, learning_rate=0.035), seed=777)
pos_reg.fit(X_tr, Y_tr, X_va, Y_va)
Y_va_r = np.clip(pos_reg.predict(X_va), 0, None)
print("Valid sMAPE | PosReg [window-level]:", round(smape_eval(Y_va, Y_va_r), 4))

# 7) (신규) Global LSTM
if USE_LSTM_GLOBAL:
    Fdim = len(FEATURE_COLS)
    lstm_global = GlobalLSTM(in_dim=Fdim)
    lstm_global.fit(X_tr, Y_tr, X_va, Y_va, sample_weight=W_TR)
    Y_va_lstm = lstm_global.predict(X_va)
    print("Valid sMAPE | LSTM Global [window-level]:", round(smape_eval(Y_va, Y_va_lstm), 4))
    # LSTM + PosReg 블렌드 → Yr 교체
    alpha = float(np.clip(LSTM_POSREG_BLEND_ALPHA, 0.0, 1.0))
    Y_va_r_blend = (1.0 - alpha) * Y_va_r + alpha * Y_va_lstm
    print("Valid sMAPE | PosReg+LSTM blend [window-level]:", round(smape_eval(Y_va, Y_va_r_blend), 4))
    Y_va_r = Y_va_r_blend  # 이후 파이프라인에서 Yr은 블렌딩 버전 사용

# ------------------------------------------------------------
# Positive-day classifier + 게이팅 스쿼시
# ------------------------------------------------------------
def make_cls_features(X):
    F = len(FEATURE_COLS)
    Z=[]
    for x in X:
        x2d = x.reshape(LOOKBACK, F)
        last = x2d[-1]
        z = np.hstack([
            last,
            x2d[-7:,0].mean(), x2d[-7:,0].max(), x2d[-7:,0].min(),
            (x2d[-7:,0]>0).mean(),
            x2d[-14:,0].mean(),
            (x2d[-14:,0]>0).mean(),
        ])
        Z.append(z.astype(np.float32))
    return np.asarray(Z, dtype=np.float32)

def squash_prob(p, a=SQUASH_A):
    return (1-a)*p + a*0.5

def make_calibrated_gb(seed=42):
    clf = GradientBoostingClassifier(random_state=seed, n_estimators=250, learning_rate=0.06, max_depth=3)
    try:
        return CalibratedClassifierCV(estimator=clf, method='isotonic', cv=3)
    except TypeError:
        return CalibratedClassifierCV(base_estimator=clf, method='isotonic', cv=3)

Z_tr, Z_va = make_cls_features(X_tr), make_cls_features(X_va)
pos_models=[]
for h in range(PREDICT):
    y_pos = (Y_tr[:,h] > 0).astype(int)
    cal = make_calibrated_gb(SEED)
    cal.fit(Z_tr, y_pos)
    pos_models.append(cal)
P_va = np.column_stack([m.predict_proba(Z_va)[:,1] for m in pos_models])

# ------------------------------------------------------------
# Validation bank (multi rolling) + 메타블렌더(제약) + 후처리
# ------------------------------------------------------------
def cap_value_for_this_menu_from_group(g_sorted: pd.DataFrame,
                                       as_of_date: pd.Timestamp,
                                       lookback: int = CAP_LOOKBACK,
                                       perc: float = CAP_PERC,
                                       mul: float = CAP_MUL,
                                       cap_min: float | None = CAP_MIN,
                                       cap_max: float | None = CAP_MAX) -> float:
    hist = g_sorted[g_sorted['영업일자'] <= as_of_date]
    if hist.empty: return float('inf')
    pos = hist.loc[hist['매출수량'] > 0, '매출수량'].astype(float)
    if pos.empty: return float('inf')
    recent = pos.tail(lookback)
    if recent.shape[0] < max(3, int(0.25 * lookback)):
        recent = pos.tail(min(pos.shape[0], lookback * 6))
    if recent.empty: return float('inf')
    cap = float(np.percentile(recent, perc)) * float(mul)
    if cap_min is not None: cap = max(cap, float(cap_min))
    if cap_max is not None: cap = min(cap, float(cap_max))
    return cap if cap > 0 else float('inf')

F = len(FEATURE_COLS)
def build_asof_arrays(df_feat_full: pd.DataFrame, as_of_date: pd.Timestamp):
    df = df_feat_full[df_feat_full['영업일자']<=as_of_date].copy()
    X, meta, lastrows, Z, caps = [], [], [], [], []
    for menu, g in df.groupby('영업장명_메뉴명'):
        g_sorted = g.sort_values('영업일자').copy()
        arr = g_sorted[FEATURE_COLS].to_numpy(dtype=np.float32)
        if arr.shape[0] < LOOKBACK: continue
        X.append(arr[-LOOKBACK:].reshape(-1))
        store = menu.split("_",1)[0].strip()
        meta.append((menu, store))
        s  = g_sorted['영업장명_메뉴명'].astype(str)
        sp = s.str.split('_', n=1, expand=True)
        g_sorted['store'] = sp[0].astype(str).str.strip()
        g_sorted['menu']  = sp[1].fillna('UNK') if 1 in sp.columns else 'UNK'
        lastrows.append(g_sorted.iloc[[-1]])
        x2d = arr[-LOOKBACK:, :]
        last = x2d[-1]
        z = np.hstack([last,
                       x2d[-7:,0].mean(), x2d[-7:,0].max(), x2d[-7:,0].min(),
                       (x2d[-7:,0]>0).mean(),
                       x2d[-14:,0].mean(),
                       (x2d[-14:,0]>0).mean()])
        Z.append(z.astype(np.float32))
        cap_val = cap_value_for_this_menu_from_group(g_sorted, as_of_date) if USE_SOFT_CAP else float('inf')
        caps.append(float(cap_val))
    if len(X)==0: return None, [], [], None, []
    return np.stack(X), meta, lastrows, np.stack(Z), caps

def predict_channels_asof(as_of_date: pd.Timestamp):
    X_te, meta, lastrows, Z_te, caps = build_asof_arrays(train_feat, as_of_date)
    if X_te is None: return []
    Yg_all  = np.clip(global_xgb_ens.predict(X_te), 0, None)
    Yl_all  = np.zeros((len(meta), PREDICT))
    for i, (menu, store) in enumerate([m for m in meta]):
        if store in local_models:
            Yl_all[i] = np.clip(local_models[store].predict(X_te[i:i+1])[0], 0, None)
    Yc_all  = np.column_stack([m.predict(X_te) for m in cat_models])
    Yh_all  = np.vstack([lgb_h.predict_lastrow(lr) for lr in lastrows])
    Yn_all  = np.stack([snaive7_from_window(x) for x in X_te])
    Yr_all  = np.clip(pos_reg.predict(X_te), 0, None)
    if USE_LSTM_GLOBAL:
        Yls_all = lstm_global.predict(X_te)
        alpha = float(np.clip(LSTM_POSREG_BLEND_ALPHA, 0.0, 1.0))
        Yr_all = (1.0 - alpha) * Yr_all + alpha * Yls_all
    else:
        Yls_all = np.zeros_like(Yr_all)
    Yq10_all = np.vstack([lgb_q10.predict_lastrow(lr) for lr in lastrows])
    Yq50_all = np.vstack([lgb_q50.predict_lastrow(lr) for lr in lastrows])
    Yq90_all = np.vstack([lgb_q90.predict_lastrow(lr) for lr in lastrows])
    return [{
        '업장': meta[i][1], '영업장명_메뉴명': meta[i][0],
        'A': np.array([ float(train_raw.loc[(train_raw['영업장명_메뉴명']==meta[i][0]) &
                                            (train_raw['영업일자']==(as_of_date + pd.Timedelta(days=h))), '매출수량'].iloc[0])
                        if len(train_raw.loc[(train_raw['영업장명_메뉴명']==meta[i][0]) &
                                             (train_raw['영업일자']==(as_of_date + pd.Timedelta(days=h)))]) else 0.0
                        for h in range(1, PREDICT+1)], dtype=float),
        'Yg': Yg_all[i], 'Yl': Yl_all[i], 'Yc': Yc_all[i], 'Yh': Yh_all[i],
        'Yn': Yn_all[i], 'Yr': Yr_all[i], 'Yls': Yls_all[i],
        'Yq10': Yq10_all[i], 'Yq50': Yq50_all[i], 'Yq90': Yq90_all[i],
        'cap': caps[i],
    } for i in range(len(meta))]

# 다중 롤링 뱅크
def build_validation_banks():
    banks=[]
    for offset in [0, 14, 28]:
        seg_ends=[]
        as_of = cut_date + pd.Timedelta(days=offset)
        while as_of + pd.Timedelta(days=PREDICT) <= train_raw['영업일자'].max():
            seg_ends.append(as_of)
            as_of += pd.Timedelta(days=PREDICT)
            if len(seg_ends) >= VALID_DAYS//PREDICT: break
        bank=[]
        for seg_idx, as_of in enumerate(tqdm(seg_ends, desc=f"Build bank offset={offset}")):
            rows = predict_channels_asof(as_of)
            X_asof, meta_asof, _, Z_te_asof, _ = build_asof_arrays(train_feat, as_of)
            if X_asof is not None:
                P_asof = np.column_stack([m.predict_proba(Z_te_asof)[:,1] for m in pos_models])
            else:
                P_asof = None
            for i, r in enumerate(rows):
                r2 = r.copy()
                r2['seg'] = seg_idx
                r2['P']   = squash_prob(P_asof[i]) if P_asof is not None else np.ones(PREDICT, dtype=float)
                bank.append(r2)
        banks.append(bank)
    return banks

val_banks = build_validation_banks()

# 메뉴 희소도 → sNaive 상한 보정
menu_sparse = {}
for bank in val_banks:
    for m in set([r['영업장명_메뉴명'] for r in bank]):
        g2 = train_feat[train_feat['영업장명_메뉴명']==m].sort_values('영업일자')
        menu_sparse[m] = float(g2.get('zero_rate7', pd.Series([0])).mean())

# 업장별 ε-floor (양수 5% 분위)
store_pos_q5 = {}
for st, g in train_raw.groupby('업장'):
    pos = g.loc[g['매출수량']>0, '매출수량'].values
    if len(pos)==0: store_pos_q5[st]=EPS_MIN_FLOOR
    else: store_pos_q5[st] = max(EPS_MIN_FLOOR, float(np.percentile(pos, 5)))

# ------------------------------------------------------------
# 후처리/메타블렌더
# ------------------------------------------------------------
def apply_postprocess(base: np.ndarray,
                      q10: np.ndarray,
                      q90: np.ndarray,
                      p_prob: np.ndarray,
                      eps_store: float,
                      cap: float,
                      q50: np.ndarray | None = None) -> np.ndarray:
    floor = np.maximum(eps_store, q10) if q50 is None else np.maximum(np.maximum(eps_store, q10), 0.5*q50)
    p_prob = squash_prob(p_prob, a=SQUASH_A)
    base = np.maximum(base, floor)
    base = base * (GATE_A + GATE_B * p_prob)
    qcap = q90 * 1.10
    cap_final = cap
    if USE_SOFT_CAP and np.isfinite(cap_final) and (cap_final > 0):
        cap_final = min(cap_final, float(np.max(qcap)))
        base = np.minimum(base, cap_final)
    return base

def _apply_post_all_channels(r, w6, eps_store):
    # w6: [Yg,Yl,Yc,Yh,Yr,Yn]  (기존 구조 유지)
    base = (w6[0]*r['Yg'] + w6[1]*r['Yl'] + w6[2]*r['Yc'] +
            w6[3]*r['Yh'] + w6[4]*r['Yr'] + w6[5]*r['Yn'])
    base = apply_postprocess(base, r['Yq10'], r['Yq90'],
                             r.get('P', np.ones_like(base)), eps_store, r.get('cap', float('inf')), q50=r.get('Yq50'))
    return np.maximum(base, 0.0)

def _project_simplex_with_cap(v, cap_idx=5, cap_val=GLOBAL_SNAIVE_CAP):
    w = np.maximum(v, 0.0)
    s = w.sum()
    if s == 0:
        w[0]=1.0; return w
    w /= s
    if w[cap_idx] > cap_val:
        overflow = w[cap_idx] - cap_val
        w[cap_idx] = cap_val
        rest_sum = w.sum() - cap_val
        if rest_sum > 0:
            w *= (1.0 - cap_val) / rest_sum
            w[cap_idx] = cap_val
    return w / w.sum()

def fit_global_constrained_weights(val_banks, store_weights=None,
                                   max_iter=200, lr=0.03, snaive_cap=GLOBAL_SNAIVE_CAP):
    # 전역 가중은 기존 6채널 체계 유지: [Yg,Yl,Yc,Yh,Yr,Yn]
    w = np.ones(6, dtype=float) / 6.0
    sw = store_weights or {}
    for _ in range(max_iter):
        grad = np.zeros_like(w)
        n = 0
        for bank in val_banks:
            for r in bank:
                A = r['A']; mask = (A > 0)
                if mask.sum()==0: continue
                eps_store = store_pos_q5.get(r['업장'], EPS_MIN_FLOOR) if USE_DYNAMIC_EPSILON else EPS_MIN_FLOOR
                P = _apply_post_all_channels(r, w, eps_store)
                C = np.column_stack([r['Yg'], r['Yl'], r['Yc'], r['Yh'], r['Yr'], r['Yn']])[mask]
                Am, Pm = A[mask], P[mask]
                g = -2.0 * (Am - Pm) @ C
                grad += g * float(sw.get(r['업장'], 1.0))
                n += 1
        if n==0: break
        w = w - lr * grad / n
        w = _project_simplex_with_cap(w, cap_idx=5, cap_val=snaive_cap)
    return w

W_GLOBAL6 = fit_global_constrained_weights(val_banks, store_weights=LB_STORE_W,
                                           max_iter=200, lr=0.03, snaive_cap=GLOBAL_SNAIVE_CAP)
print("Global constrained weights [Yg,Yl,Yc,Yh,Yr,Yn]:", np.round(W_GLOBAL6, 3))

def estimate_store_h_bias(banks):
    vals = {}
    stores = {r['업장'] for b in banks for r in b}
    for s in stores:
        for h in range(PREDICT):
            vals[(s,h)] = []
    for bank in banks:
        for r in bank:
            eps_store = store_pos_q5.get(r['업장'], EPS_MIN_FLOOR) if USE_DYNAMIC_EPSILON else EPS_MIN_FLOOR
            P = _apply_post_all_channels(r, W_GLOBAL6, eps_store)
            for h in range(PREDICT):
                A = float(r['A'][h]); Ph = float(P[h])
                if A > 0 and Ph > 0:
                    vals[(r['업장'], h)].append(A/Ph)
    bias = {}
    for k, arr in vals.items():
        if len(arr) >= 10:
            m = float(np.median(arr))
            bias[k] = float(np.clip(m, 0.88, 1.18))  # Winsorize ±12%
        else:
            bias[k] = 1.0
    return bias

STORE_H_BIAS = estimate_store_h_bias(val_banks)

def menu_objective_exact(menu, weights5, bank, alpha_by_h):
    # weights5: [Yg,Yl,Yc,Yh,Yr]  (Yn은 잔여)
    wg, wl, wc, wh, wr = weights5
    wn = 1.0 - (wg + wl + wc + wh + wr)
    if wn < 0:
        return 1e9
    cap_snaive = float(np.clip(SNAIVE_MAX_W + 0.20 * menu_sparse.get(menu, 0.0),
                               SNAIVE_MAX_W, SNAIVE_CAP_MAX))
    if wn > cap_snaive:
        return 1e9
    rows = [r for r in bank if r['영업장명_메뉴명'] == menu]
    if not rows:
        return 1e9
    store = rows[0]['업장']
    A_all, P_all = [], []
    for r in rows:
        base = (wg * r['Yg'] + wl * r['Yl'] + wc * r['Yc'] +
                wh * r['Yh'] + wr * r['Yr'] + wn * r['Yn']).copy()
        base = np.array([alpha_by_h.get(h, 1.0) * base[h] for h in range(PREDICT)], dtype=float)
        eps_store = store_pos_q5.get(store, EPS_MIN_FLOOR) if USE_DYNAMIC_EPSILON else EPS_MIN_FLOOR
        base = apply_postprocess(
            base, r['Yq10'], r['Yq90'], r.get('P', np.ones(PREDICT)),
            eps_store, r.get('cap', float('inf')), q50=r.get('Yq50')
        )
        for h in range(PREDICT):
            base[h] *= STORE_H_BIAS.get((store, h), 1.0)
        A_all.append(r['A'])
        P_all.append(base)
    A_all = np.concatenate(A_all)
    P_all = np.concatenate(P_all)
    s_i, Ti = smape_ignore_zeros_per_item(A_all, P_all)
    return s_i if not np.isnan(s_i) else 1e9

def grid_search_menu_exact(menu, bank, step=BLEND_STEP):
    grid = np.arange(0, 1.0+1e-9, step)
    best=(1,0,0,0,0); best_s=1e9
    alpha_init = {h:1.0 for h in range(PREDICT)}
    for wg in grid:
      for wl in grid:
        for wc in grid:
          for wh in grid:
            for wr in grid:
                s = wg+wl+wc+wh+wr
                if s > 1.0: continue
                val = menu_objective_exact(menu, (wg,wl,wc,wh,wr), bank, alpha_init)
                if val < best_s: best_s, best = val, (float(wg),float(wl),float(wc),float(wh),float(wr))
    return best, best_s

def refine_cd_menu_exact(menu, bank, w0, step=CD_REFINE_STEP, rounds=CD_REFINE_ROUNDS, alpha=True):
    w = list(w0); alpha_by_h = {h:1.0 for h in range(PREDICT)}
    for _ in range(rounds):
        for k in range(5):
            cur = w.copy()
            trial=[]
            for delta in [-step, 0.0, step]:
                w2 = cur.copy()
                w2[k] = float(np.clip(w2[k]+delta, 0.0, 1.0))
                if sum(w2) > 1.0: continue
                val = menu_objective_exact(menu, tuple(w2), bank, alpha_by_h)
                trial.append((val, w2))
            if trial:
                trial.sort(key=lambda x: x[0]); w = trial[0][1]
    if alpha:
        base_w = tuple(w)
        for h in range(PREDICT):
            best_a, best_s = 1.0, 1e9
            for a in np.arange(0.92, 1.10+1e-9, 0.02):
                a_map = {k:(alpha_by_h.get(k,1.0) if k!=h else float(a)) for k in range(PREDICT)}
                s = menu_objective_exact(menu, base_w, bank, a_map)
                if s < best_s: best_s, best_a = s, float(a)
            alpha_by_h[h] = best_a
    return tuple(w), alpha_by_h

# 메뉴별 가중 탐색 (bank0 기준) + 전역 가중 수축
def shrink_menu_weights(w_menu_hat, w_global6, menu, lam=8.0):
    wg, wl, wc, wh, wr = w_menu_hat
    wn_hat = max(0.0, 1.0 - (wg+wl+wc+wh+wr))
    w_hat6 = np.array([wg,wl,wc,wh,wr,wn_hat], dtype=float)
    sparsity = menu_sparse.get(menu, 0.0)
    lam_eff = lam * (1.0 + 2.0*sparsity)
    n_eff = 20.0
    alpha = n_eff/(n_eff+lam_eff)
    w_blend = alpha*w_hat6 + (1.0-alpha)*w_global6
    w_blend = _project_simplex_with_cap(w_blend, cap_idx=5, cap_val=GLOBAL_SNAIVE_CAP)
    return tuple(w_blend[:5]), float(w_blend[5])

weights5_raw = {}
alpha_by_menu_h = {}
first_bank = val_banks[0]
for menu in tqdm(sorted(set([r['영업장명_메뉴명'] for r in first_bank])), desc="Per-menu weight search (Grid+CD on bank0)"):
    w0, _ = grid_search_menu_exact(menu, first_bank)
    w1, a1 = refine_cd_menu_exact(menu, first_bank, w0, alpha=True)
    weights5_raw[menu] = w1
    alpha_by_menu_h[menu] = a1

def build_val_prediction_long(banks):
    rows_out=[]
    for bank in banks:
        menus = sorted(set([r['영업장명_메뉴명'] for r in bank]))
        for menu in menus:
            if USE_PER_MENU_WEIGHTS:
                w5_raw = weights5_raw.get(menu, (1,0,0,0,0))
                w5, wn_extra = shrink_menu_weights(w5_raw, W_GLOBAL6, menu, lam=8.0)
            else:
                wg, wl, wc, wh, wr, wn_extra = [float(x) for x in W_GLOBAL6]
                w5 = (wg, wl, wc, wh, wr)
            cap_snaive = float(np.clip(SNAIVE_MAX_W + 0.20*menu_sparse.get(menu,0.0),
                                       SNAIVE_MAX_W, SNAIVE_CAP_MAX))
            wn_extra = float(np.clip(wn_extra, 0.0, cap_snaive))
            a_map = alpha_by_menu_h.get(menu, {h:1.0 for h in range(PREDICT)})
            for r in [x for x in bank if x['영업장명_메뉴명']==menu]:
                base = (w5[0]*r['Yg'] + w5[1]*r['Yl'] + w5[2]*r['Yc']
                        + w5[3]*r['Yh'] + w5[4]*r['Yr'] + wn_extra*r['Yn']).copy()
                base = np.array([a_map.get(h,1.0)*base[h] for h in range(PREDICT)], dtype=float)
                eps_store = store_pos_q5.get(r['업장'], EPS_MIN_FLOOR) if USE_DYNAMIC_EPSILON else EPS_MIN_FLOOR
                base = apply_postprocess(base, r['Yq10'], r['Yq90'], r.get('P', np.ones(PREDICT)),
                                         eps_store, r.get('cap', float('inf')), q50=r.get('Yq50'))
                for h in range(PREDICT):
                    base[h] *= STORE_H_BIAS.get((r['업장'], h), 1.0)
                for t in range(PREDICT):
                    rows_out.append({
                        '업장': r['업장'],
                        '영업장명_메뉴명': r['영업장명_메뉴명'],
                        'seg': r['seg'],
                        't': t+1,
                        'A': float(r['A'][t]),
                        'P': float(max(base[t], 0.0)),
                    })
    return pd.DataFrame(rows_out)

val_long = build_val_prediction_long(val_banks)
val_exact = leaderboard_smape(val_long, store_weights=LB_STORE_W)
print(f"[EXACT] Leaderboard-like sMAPE (multi-bank avg, 업장가중 & A=0 제외): {val_exact:.4f}")

# ------------------------------------------------------------
# Test inference & Submission
# ------------------------------------------------------------
def make_features_test(df_raw):
    df = clean_df(df_raw)
    df = add_features(df)
    s = df['영업장명_메뉴명'].astype(str); sp = s.str.split('_', n=1, expand=True)
    df['store'] = sp[0]; df['menu'] = sp[1].fillna('UNK')
    return df

def build_test_windows_one_file(path: str):
    df = make_features_test(read_csv_safe(path))
    X, meta, lastrows, Z, caps = [], [], [], [], []
    for menu, g in df.groupby('영업장명_메뉴명'):
        g_sorted = g.sort_values('영업일자')
        arr = g_sorted[FEATURE_COLS].to_numpy(dtype=np.float32)
        if arr.shape[0] < LOOKBACK: continue
        X.append(arr[-LOOKBACK:].reshape(-1))
        store = menu.split("_",1)[0].strip()
        m = re.search(r'(TEST_\d+)', os.path.basename(path))
        prefix = m.group(1) if m else os.path.basename(path).split('.')[0]
        meta.append((prefix, menu, store))
        lastrows.append(g_sorted.iloc[[-1]])
        x2d = arr[-LOOKBACK:, :]
        last = x2d[-1]
        z = np.hstack([last,
                       x2d[-7:,0].mean(), x2d[-7:,0].max(), x2d[-7:,0].min(),
                       (x2d[-7:,0]>0).mean(),
                       x2d[-14:,0].mean(),
                       (x2d[-14:,0]>0).mean()])
        Z.append(z.astype(np.float32))
        if USE_SOFT_CAP:
            as_of = g_sorted['영업일자'].max()
            cap_val = cap_value_for_this_menu_from_group(g_sorted, as_of)
        else:
            cap_val = float('inf')
        caps.append(float(cap_val))
    return (np.stack(X), meta, lastrows, np.stack(Z), caps) if X else (None, [], [], None, [])

def predict_one_file_hybrid(path: str):
    X_te, meta, lastrows, Z_te, caps = build_test_windows_one_file(path)
    rows=[]
    if X_te is None: return pd.DataFrame(rows)
    Yg_all  = np.clip(global_xgb_ens.predict(X_te), 0, None)
    Yl_all  = np.zeros((len(meta), PREDICT))
    for i, (_, menu, store) in enumerate(meta):
        if store in local_models:
            Yl_all[i] = np.clip(local_models[store].predict(X_te[i:i+1])[0], 0, None)
    Yc_all  = np.column_stack([m.predict(X_te) for m in cat_models])
    Yh_all  = np.vstack([lgb_h.predict_lastrow(lr) for lr in lastrows])
    Yn_all  = np.stack([snaive7_from_window(x) for x in X_te])
    Yr_all  = np.clip(pos_reg.predict(X_te), 0, None)
    if USE_LSTM_GLOBAL:
        Yls_all = lstm_global.predict(X_te)
        alpha = float(np.clip(LSTM_POSREG_BLEND_ALPHA, 0.0, 1.0))
        Yr_all = (1.0 - alpha) * Yr_all + alpha * Yls_all
    else:
        Yls_all = np.zeros_like(Yr_all)
    Yq10_all = np.vstack([lgb_q10.predict_lastrow(lr) for lr in lastrows])
    Yq50_all = np.vstack([lgb_q50.predict_lastrow(lr) for lr in lastrows])
    Yq90_all = np.vstack([lgb_q90.predict_lastrow(lr) for lr in lastrows])

    if Z_te is not None:
        P_te = np.column_stack([m.predict_proba(Z_te)[:,1] for m in pos_models])
        P_te = squash_prob(P_te, a=SQUASH_A)
    else:
        P_te = np.ones((len(meta), PREDICT), dtype=float)

    for i, (prefix, menu, store) in enumerate(meta):
        # 메뉴-전용 가중은 기존 5채널(Yg,Yl,Yc,Yh,Yr) + 잔여(Yn) 구조 유지 (Yr은 이미 LSTM 블렌딩)
        if USE_PER_MENU_WEIGHTS:
            w5_raw = weights5_raw.get(menu, (1.0,0.0,0.0,0.0,0.0))
            w5, wn = shrink_menu_weights(w5_raw, W_GLOBAL6, menu, lam=8.0)
        else:
            wg, wl, wc, wh, wr, wn = [float(x) for x in W_GLOBAL6]
            w5 = (wg, wl, wc, wh, wr)
        cap_snaive = float(np.clip(SNAIVE_MAX_W + 0.20*menu_sparse.get(menu,0.0),
                                   SNAIVE_MAX_W, SNAIVE_CAP_MAX))
        wn = float(np.clip(wn, 0.0, cap_snaive))

        base = (w5[0]*Yg_all[i] + w5[1]*Yl_all[i] + w5[2]*Yc_all[i]
                + w5[3]*Yh_all[i] + w5[4]*Yr_all[i] + wn*Yn_all[i])

        a_map = alpha_by_menu_h.get(menu, {h:1.0 for h in range(PREDICT)})
        base = np.array([a_map.get(h,1.0)*base[h] for h in range(PREDICT)], dtype=float)

        eps_store = store_pos_q5.get(store, EPS_MIN_FLOOR) if USE_DYNAMIC_EPSILON else EPS_MIN_FLOOR
        cap_val   = caps[i] if (i < len(caps)) else float('inf')
        base = apply_postprocess(base, Yq10_all[i], Yq90_all[i], P_te[i], eps_store, cap_val, q50=Yq50_all[i])

        for h in range(PREDICT):
            base[h] *= STORE_H_BIAS.get((store, h), 1.0)

        if ALL_ZERO_GUARD and np.allclose(base, 0.0):
            floor = np.maximum(eps_store, Yq10_all[i])
            base  = np.maximum(Yn_all[i], floor)

        for h, val in enumerate(np.maximum(base,0.0), 1):
            rows.append({
                '영업일자': f'{prefix}+{h}일',
                '영업장명_메뉴명': menu,
                '매출수량': float(val)
            })
    return pd.DataFrame(rows)

# 실행
test_dir_candidates = [TEST_DIR, os.path.join(TEST_DIR, 'test')]
test_files=[]
for td in test_dir_candidates:
    if os.path.isdir(td):
        test_files = sorted([p for p in glob.glob(os.path.join(td, 'TEST_*.csv')) if os.path.isfile(p)])
        if len(test_files)>0: break

pred_list = [predict_one_file_hybrid(p) for p in tqdm(test_files, desc="Predict test files")]
pred_df = pd.concat(pred_list, ignore_index=True) if len(pred_list) else pd.DataFrame(columns=['영업일자','영업장명_메뉴명','매출수량'])
print("Test files:", len(test_files), "| preds rows:", len(pred_df))

def to_submission(pred_df: pd.DataFrame, sample: pd.DataFrame):
    key_norm = list(zip(pred_df['영업일자'].astype(str),
                        pred_df['영업장명_메뉴명'].map(normalize_name)))
    pred_map = dict(zip(key_norm, pred_df['매출수량']))
    out = sample.copy()
    for i in out.index:
        date = str(out.loc[i, '영업일자'])
        for col in out.columns[1:]:
            col_norm = normalize_name(col)
            out.loc[i, col] = float(pred_map.get((date, col_norm), 0.0))
    return out

if os.path.exists(SAMPLE_SUB):
    sample = read_csv_safe(SAMPLE_SUB)
    submission = to_submission(pred_df, sample)
else:
    menus = sorted(pred_df['영업장명_메뉴명'].unique().tolist())
    dates = sorted(pred_df['영업일자'].unique().tolist())
    submission = pd.DataFrame({'영업일자': dates})
    for m in menus:
        tmp = pred_df[pred_df['영업장명_메뉴명']==m][['영업일자','매출수량']]
        submission = submission.merge(tmp, how='left', on='영업일자').rename(columns={'매출수량': m})
    submission.fillna(0.0, inplace=True)

submission.to_csv(SAVE_PATH, index=False, encoding='utf-8-sig')
print("Saved:", SAVE_PATH)

# 품질 점검
try:
    sub = pd.read_csv(SAVE_PATH)
    vals = sub.iloc[:,1:].to_numpy(dtype=float) if sub.shape[1]>1 else np.array([])
    all_zero_cols = [c for c in sub.columns[1:] if np.all(np.isclose(sub[c].to_numpy(dtype=float), 0.0))]
    print("All-zero menu columns:", len(all_zero_cols))
    print("Submission shape:", sub.shape)
    if vals.size>0:
        print("positives:", int((vals>0).sum()), "| zeros:", int((vals==0).sum()))
        print("dist | min/p10/p50/p90/max:",
              float(np.nanmin(vals)),
              float(np.nanpercentile(vals,10)),
              float(np.nanpercentile(vals,50)),
              float(np.nanpercentile(vals,90)),
              float(np.nanmax(vals)))
except Exception as e:
    print("Submission sanity check failed:", e)


Mounted at /content/drive
Versions | xgboost: 3.0.4 | lightgbm: 4.6.0 | GPU: True
Train: 2023-01-01 ~ 2024-06-15 | Valid from: 2024-04-20
n_features: 36
Train/Valid windows: 85306 10808
Sample weight | min/mean/max: 0.75 1.3511817455291748 2.6599998474121094
Valid sMAPE | Global XGB ens [window-level]: 0.6591


Local XGB (by store):   0%|          | 0/9 [00:00<?, ?it/s]

Local XGB models: 9
Valid sMAPE | Global CAT (Poisson) [window-level]: 0.5667
Valid sMAPE | LGB horizon [window-level]: 0.7433
Valid sMAPE | PosReg [window-level]: 0.5435
Valid sMAPE | LSTM Global [window-level]: 0.7582
Valid sMAPE | PosReg+LSTM blend [window-level]: 0.5603


Build bank offset=0:   0%|          | 0/8 [00:00<?, ?it/s]

Build bank offset=14:   0%|          | 0/6 [00:00<?, ?it/s]

...

Global constrained weights [Yg,Yl,Yc,Yh,Yr,Yn]: [0.079 0.047 0.108 0.532 0.124 0.111]


Per-menu weight search (Grid+CD on bank0):   0%|          | 0/193 [00:00<?, ?it/s]

[EXACT] Leaderboard-like sMAPE (multi-bank avg, 업장가중 & A=0 제외): 0.3646


Predict test files:   0%|          | 0/10 [00:00<?, ?it/s]

Test files: 10 | preds rows: 13510
Saved: /content/drive/MyDrive/data/lg/submission_robust_opt_lstm.csv
All-zero menu columns: 0
Submission shape: (70, 194)
positives: 13510 | zeros: 0
dist | min/p10/p50/p90/max: 0.1360013172446222 0.9865838892559712 1.7040638231394576 21.115789926567125 664.1903032838068
